# FlowDesk Analytics — Notebook 4
## A/B Testing — Did Our Experiments Work and Should We Ship?

**Author:** Siddharth Bokka  
**Project:** FlowDesk B2B SaaS Analytics Portfolio  

---

### Framing

> *"p < 0.05 is the beginning of the analysis, not the end of it. At Meta, Airbnb, and LinkedIn, shipping a feature requires answering: Did it improve the primary metric? Did it harm any guardrail metrics? Was the effect consistent across segments? Could it be a novelty effect? Only then do you ship."*

This notebook covers the **complete, production-grade A/B testing workflow** — not just statistical significance testing.

### Experiments Analyzed

1. **`onboarding_checklist`** — Interactive checklist shown on signup
   - *Hypothesis:* Reduces time-to-activate and improves D7 retention
   - *Primary metric:* `d7_retained` (binary)
   - *Guardrail metric:* `is_activated`

2. **`annual_pricing_nudge`** — "Save 30% with annual plan" banner on upgrade page
   - *Hypothesis:* Increases annual plan upgrades and revenue per user
   - *Primary metric:* `revenue` (continuous)
   - *Guardrail metric:* `is_activated` ← **This is where the conflict emerges**

### The A/B Testing Framework We Use

Every experiment analysis follows this checklist:
1. **Sanity Checks** — Sample balance, temporal distribution, pre-experiment comparability
2. **Primary Metric Analysis** — Effect size, confidence intervals, significance test
3. **Guardrail Metric Analysis** — Did anything break?
4. **Power Analysis (Retrospective)** — Was the experiment adequately powered?
5. **Novelty Effect Check** — Does the effect persist over time?
6. **Segment Analysis (HTE)** — Is the effect consistent across user groups?
7. **Ship / No-Ship Recommendation**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import norm, proportions_ztest, ttest_ind
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'figure.figsize': (10, 5),
})
sns.set_palette('tab10')

# Load datasets
users         = pd.read_parquet('../data/users.parquet')
events        = pd.read_parquet('../data/events.parquet')
feature_usage = pd.read_parquet('../data/feature_usage.parquet')
experiments   = pd.read_parquet('../data/experiments.parquet')
workspaces    = pd.read_parquet('../data/workspaces.parquet')
tickets       = pd.read_parquet('../data/support_tickets.parquet')

users['signup_date']      = pd.to_datetime(users['signup_date'])
events['event_ts']        = pd.to_datetime(events['event_ts'])
experiments['assigned_at'] = pd.to_datetime(experiments['assigned_at'])

print(f"Users: {len(users):,} | Events: {len(events):,} | Experiments: {len(experiments):,}")
print(f"Experiment names: {experiments['experiment_name'].unique()}")
print(f"Variants: {experiments['variant'].unique()}")

---
### Helper Functions

We define reusable functions for the statistical tests so each experiment analysis is consistent and reproducible.

In [ ]:
def wilson_ci(successes, n, alpha=0.05):
    """Wilson score confidence interval for a proportion."""
    z = norm.ppf(1 - alpha/2)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    spread = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return center - spread, center + spread


def two_proportion_ztest(n1, p1, n2, p2, alpha=0.05):
    """
    Two-proportion z-test.
    n1, n2: sample sizes
    p1, p2: observed proportions (treatment, control)
    Returns: z_stat, p_value, significant (bool), abs_lift_pp, rel_lift_pct
    """
    count = np.array([p1*n1, p2*n2])
    nobs  = np.array([n1, n2])
    z_stat, p_value = proportions_ztest(count, nobs)
    abs_lift  = (p1 - p2) * 100       # in percentage points
    rel_lift  = (p1 - p2) / p2 * 100  # relative %
    return {
        'n_treatment': n1, 'n_control': n2,
        'p_treatment': p1, 'p_control': p2,
        'z_statistic': z_stat, 'p_value': p_value,
        'significant': p_value < alpha,
        'abs_lift_pp': abs_lift,
        'rel_lift_pct': rel_lift,
    }


def power_analysis_proportion(p_control, mde_pp, alpha=0.05, power=0.80):
    """
    Minimum sample size per variant for a two-proportion z-test.
    p_control: baseline proportion
    mde_pp: minimum detectable effect in percentage points
    """
    p_treatment = p_control + mde_pp / 100
    p_bar = (p_control + p_treatment) / 2
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta  = norm.ppf(power)
    n = (2 * (z_alpha + z_beta)**2 * p_bar * (1 - p_bar)) / ((p_treatment - p_control)**2)
    return int(np.ceil(n))


def mde_from_n(n, p_control, alpha=0.05, power=0.80):
    """What's the MDE we can detect given our actual sample size n per variant?"""
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta  = norm.ppf(power)
    # Solve n = 2*(z_a+z_b)^2 * p*(1-p) / delta^2  →  delta = sqrt(2*(z_a+z_b)^2*p*(1-p)/n)
    delta = np.sqrt(2 * (z_alpha + z_beta)**2 * p_control * (1 - p_control) / n)
    return delta * 100  # in percentage points


def print_result(result_dict):
    sig = "YES ✓" if result_dict['significant'] else "NO ✗"
    print(f"  Treatment:  n={result_dict['n_treatment']:,}   rate={result_dict['p_treatment']*100:.2f}%")
    print(f"  Control:    n={result_dict['n_control']:,}   rate={result_dict['p_control']*100:.2f}%")
    print(f"  z-statistic: {result_dict['z_statistic']:.3f}")
    print(f"  p-value:     {result_dict['p_value']:.4f}")
    print(f"  Significant at α=0.05: {sig}")
    print(f"  Absolute lift: {result_dict['abs_lift_pp']:+.2f} pp")
    print(f"  Relative lift: {result_dict['rel_lift_pct']:+.1f}%")


print("Helper functions defined successfully.")

---
## Section 2 — Experiment 1: Onboarding Checklist

**What was tested:** An interactive onboarding checklist shown immediately after signup, guiding new users through 5 key actions: creating their first project, inviting a teammate, creating a task, setting a due date, and exploring the dashboard view.

**Hypothesis:** Users who complete the checklist will reach the "aha moment" faster, resulting in higher D7 retention compared to the control group (no checklist).

**Primary metric:** `d7_retained` (was the user still active 7 days after signup?)  
**Guardrail metric:** `is_activated` (did the checklist overwhelm new users and *lower* activation?)

In [ ]:
# Filter to experiment 1
exp1 = experiments[experiments['experiment_name'] == 'onboarding_checklist'].copy()

print(f"Onboarding Checklist Experiment")
print(f"  Total assignments: {len(exp1):,}")
print(f"  Date range: {exp1['assigned_at'].min().date()} → {exp1['assigned_at'].max().date()}")
print(f"  Variant counts:")
print(exp1['variant'].value_counts().to_string())

### 2a. Sanity Checks — Always First

Before looking at any results, we verify the experiment was run correctly:
1. **Sample balance** — Is the split approximately 50/50?
2. **Temporal distribution** — Were assignments balanced over time, or did one variant get more users in a certain period?
3. **Pre-experiment comparability** — Are the two groups comparable on observable characteristics?

In [ ]:
# 1. Sample balance
variant_counts = exp1['variant'].value_counts()
total = len(exp1)
print("=" * 50)
print("SANITY CHECK 1: Sample Balance")
print("=" * 50)
for v, cnt in variant_counts.items():
    print(f"  {v:12s}: {cnt:,} ({cnt/total*100:.1f}%)")

# Chi-squared test for balance
expected = total / len(variant_counts)
chi2, p_balance = stats.chisquare(variant_counts.values)
print(f"  Chi-squared test: χ²={chi2:.3f}, p={p_balance:.4f}")
print(f"  Balance check: {'PASS ✓ (p > 0.05)' if p_balance > 0.05 else 'FAIL ✗ (p < 0.05 — imbalanced)'}") 

# 2. Temporal distribution
exp1['week'] = exp1['assigned_at'].dt.to_period('W').astype(str)
temporal = exp1.groupby(['week', 'variant']).size().unstack(fill_value=0)

print("\n" + "=" * 50)
print("SANITY CHECK 2: Temporal Distribution (weekly)")
print("=" * 50)
print(temporal.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# --- Sample balance pie ---
axes[0].pie(variant_counts, labels=variant_counts.index, autopct='%1.1f%%',
            colors=['#2563eb', '#dc2626'], startangle=90)
axes[0].set_title('Variant Assignment Balance\n(Should be ~50/50)', fontsize=11, fontweight='bold')

# --- Temporal distribution ---
if 'control' in temporal.columns and 'treatment' in temporal.columns:
    x_weeks = range(len(temporal))
    axes[1].plot(x_weeks, temporal['control'],   lw=2, color='#2563eb', marker='o', markersize=4, label='Control')
    axes[1].plot(x_weeks, temporal['treatment'], lw=2, color='#dc2626', marker='s', markersize=4, label='Treatment')
    axes[1].set_xticks(x_weeks)
    axes[1].set_xticklabels(temporal.index, rotation=45, ha='right', fontsize=7)
    axes[1].set_ylabel('Assignments per Week')
    axes[1].set_title('Assignment Volume Over Time\n(Should track closely — no temporal bias)',
                      fontsize=11, fontweight='bold')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.25)

plt.suptitle('Experiment 1: Onboarding Checklist — Sanity Checks', fontsize=13, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Pre-experiment comparability — join with users table to check device/plan distribution
exp1_users = exp1.merge(users[['user_id', 'primary_device', 'plan', 'acquisition_channel']], 
                        on='user_id', how='left')

print("=" * 55)
print("SANITY CHECK 3: Pre-Experiment Covariate Balance")
print("=" * 55)

for covariate in ['primary_device', 'plan']:
    print(f"\n  {covariate.upper()} distribution by variant (row %):")
    cross = pd.crosstab(exp1_users['variant'], exp1_users[covariate], normalize='index') * 100
    print(cross.round(1).to_string())
    
    # Chi-squared test for independence
    ct_raw = pd.crosstab(exp1_users['variant'], exp1_users[covariate])
    chi2, p, dof, _ = stats.chi2_contingency(ct_raw)
    print(f"  Chi² test: χ²={chi2:.2f}, p={p:.4f} — {'Groups are COMPARABLE ✓' if p > 0.05 else 'Groups differ on this covariate ⚠️'}")

### 2b. Primary Metric Analysis: D7 Retention

Now we look at the actual result. We use a **two-proportion z-test** with 95% confidence (α = 0.05).

**H₀:** p_treatment = p_control (the checklist has no effect on D7 retention)  
**H₁:** p_treatment ≠ p_control (two-tailed)

In [ ]:
control_exp1   = exp1[exp1['variant'] == 'control']
treatment_exp1 = exp1[exp1['variant'] == 'treatment']

# D7 retention stats
n_ctrl = len(control_exp1)
n_trt  = len(treatment_exp1)
p_ctrl_d7 = control_exp1['d7_retained'].mean()
p_trt_d7  = treatment_exp1['d7_retained'].mean()

# Wilson CIs
ctrl_lo, ctrl_hi = wilson_ci(control_exp1['d7_retained'].sum(),   n_ctrl)
trt_lo,  trt_hi  = wilson_ci(treatment_exp1['d7_retained'].sum(), n_trt)

print("=" * 55)
print("PRIMARY METRIC: D7 Retention Rate")
print("=" * 55)
print(f"  Control:   {p_ctrl_d7*100:.2f}%  95% CI: [{ctrl_lo*100:.2f}%, {ctrl_hi*100:.2f}%]")
print(f"  Treatment: {p_trt_d7*100:.2f}%  95% CI: [{trt_lo*100:.2f}%, {trt_hi*100:.2f}%]")

result_d7 = two_proportion_ztest(n_trt, p_trt_d7, n_ctrl, p_ctrl_d7)
print()
print_result(result_d7)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

variants  = ['Control', 'Treatment']
rates     = [p_ctrl_d7 * 100, p_trt_d7 * 100]
ci_lo     = [ctrl_lo * 100, trt_lo * 100]
ci_hi     = [ctrl_hi * 100, trt_hi * 100]
errors    = [[r - lo for r, lo in zip(rates, ci_lo)],
             [hi - r  for r, hi in zip(rates, ci_hi)]]
bar_colors = ['#6b7280', '#2563eb']

bars = ax.bar(variants, rates, color=bar_colors, width=0.45, zorder=3, alpha=0.85)
ax.errorbar(variants, rates, yerr=errors, fmt='none', color='black', capsize=8, lw=2, capthick=2, zorder=4)

for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{rate:.2f}%', ha='center', fontsize=11, fontweight='bold')

# Annotate lift
lift_pp  = result_d7['abs_lift_pp']
lift_rel = result_d7['rel_lift_pct']
sig_text = "p < 0.05 ✓" if result_d7['significant'] else f"p = {result_d7['p_value']:.3f} ✗"
ax.annotate(
    f'+{lift_pp:.2f} pp ({lift_rel:+.1f}% rel.)\n{sig_text}',
    xy=(1, p_trt_d7 * 100 - 3),
    xytext=(1.25, (p_ctrl_d7 + p_trt_d7) / 2 * 100),
    fontsize=10, color='#16a34a' if result_d7['significant'] else '#dc2626', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#16a34a' if result_d7['significant'] else '#dc2626', lw=1.5),
)

ax.set_ylabel('D7 Retention Rate (%)')
ax.set_title('Experiment 1: Onboarding Checklist\nPrimary Metric — D7 Retention (with 95% CI)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, max(rates) * 1.35)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.grid(axis='y', alpha=0.25, zorder=0)
plt.tight_layout()
plt.show()

### 2c. Guardrail Metric: Activation Rate

The checklist could have an **unintended negative effect**: if it's too long or overwhelming, new users might abandon onboarding entirely, resulting in *lower* activation rates. We check this guardrail before recommending a ship.

In [ ]:
p_ctrl_act = control_exp1['is_activated'].mean()
p_trt_act  = treatment_exp1['is_activated'].mean()

act_lo_ctrl, act_hi_ctrl = wilson_ci(control_exp1['is_activated'].sum(),   n_ctrl)
act_lo_trt,  act_hi_trt  = wilson_ci(treatment_exp1['is_activated'].sum(), n_trt)

result_act = two_proportion_ztest(n_trt, p_trt_act, n_ctrl, p_ctrl_act)

print("=" * 55)
print("GUARDRAIL METRIC: Activation Rate")
print("=" * 55)
print(f"  Control:   {p_ctrl_act*100:.2f}%  95% CI: [{act_lo_ctrl*100:.2f}%, {act_hi_ctrl*100:.2f}%]")
print(f"  Treatment: {p_trt_act*100:.2f}%  95% CI: [{act_lo_trt*100:.2f}%, {act_hi_trt*100:.2f}%]")
print()
print_result(result_act)

if result_act['abs_lift_pp'] < 0 and result_act['significant']:
    print("\n⚠️  GUARDRAIL VIOLATED: Activation rate is significantly LOWER in treatment.")
    print("    The checklist appears to be harming activation — investigate before shipping.")
elif result_act['abs_lift_pp'] < -1.0:
    print("\n⚠️  GUARDRAIL WARNING: Activation rate is directionally lower in treatment.")
    print(f"    Effect is not significant (p={result_act['p_value']:.3f}) but watch this closely.")
else:
    print("\n✓  GUARDRAIL CLEAR: Activation rate is not meaningfully degraded in treatment.")

### 2d. Sample Size and Power Analysis (Retrospective)

A common mistake in A/B testing is to look at results without asking: *"Was the experiment even capable of detecting the effect we care about?"*

We perform a **retrospective power analysis**: given our actual sample size, what is the **minimum detectable effect (MDE)** we were powered to see? And conversely, how many users would we have needed to detect a specific target effect?

In [ ]:
# What MDE can we detect with our actual sample size?
n_per_variant = min(n_ctrl, n_trt)
actual_mde    = mde_from_n(n_per_variant, p_ctrl_d7, alpha=0.05, power=0.80)

print("=" * 60)
print("POWER ANALYSIS — Experiment 1: D7 Retention")
print("=" * 60)
print(f"  Baseline (control) D7 rate:     {p_ctrl_d7*100:.2f}%")
print(f"  Observed treatment D7 rate:     {p_trt_d7*100:.2f}%")
print(f"  Observed effect:                {result_d7['abs_lift_pp']:+.2f} pp")
print()
print(f"  Sample size per variant:        {n_per_variant:,}")
print(f"  MDE detectable at 80% power:    ±{actual_mde:.2f} pp")
print()

# What n do we need for various target MDEs?
print("  Sample size needed to detect given MDE (α=0.05, power=80%):")
for mde_pp in [1, 2, 3, 5]:
    n_needed = power_analysis_proportion(p_ctrl_d7, mde_pp)
    status   = "✓ (have enough)" if n_per_variant >= n_needed else "✗ (need more)"
    print(f"    MDE = {mde_pp} pp  →  n = {n_needed:,} per variant  {status}")

print()
if actual_mde <= abs(result_d7['abs_lift_pp']):
    print("  CONCLUSION: Experiment is adequately powered to detect the observed effect.")
else:
    print("  CONCLUSION: Observed effect is below the MDE — experiment may be underpowered.")
    print("  Consider running longer before drawing conclusions.")

In [ ]:
# Power curve: required n per variant vs MDE
mde_range = np.linspace(0.5, 10, 100)
n_required = [power_analysis_proportion(p_ctrl_d7, mde) for mde in mde_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mde_range, n_required, lw=2.2, color='#2563eb')
ax.axhline(n_per_variant, color='#16a34a', lw=1.8, ls='--',
           label=f'Our sample: {n_per_variant:,} per variant')
ax.axvline(actual_mde, color='#f97316', lw=1.8, ls=':',
           label=f'Our MDE: {actual_mde:.2f} pp')
ax.axvline(abs(result_d7['abs_lift_pp']), color='#dc2626', lw=1.8, ls='-.',
           label=f'Observed effect: {abs(result_d7["abs_lift_pp"]):.2f} pp')

ax.fill_betweenx(
    [0, n_per_variant],
    [actual_mde, actual_mde],
    [10, 10],
    alpha=0.08, color='#16a34a', label='Detectable region'
)

ax.set_xlabel('Minimum Detectable Effect (percentage points)')
ax.set_ylabel('Required Sample Size (per variant)')
ax.set_title('Power Curve — Required n vs MDE\n(α=0.05, Power=80%, D7 Retention baseline)',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, min(max(n_required), n_per_variant * 3))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### 2e. Novelty Effect Check

A **novelty effect** occurs when users respond positively to a new feature simply because it's new — not because it's genuinely better. This effect typically fades after a few weeks as the novelty wears off.

To check for this, we compute D7 retention by variant **for each week of the experiment**. If the treatment advantage shrinks or disappears over time, we suspect a novelty effect.

In [ ]:
exp1['week_of_experiment'] = exp1['assigned_at'].dt.to_period('W').astype(str)

weekly_retention = (
    exp1
    .groupby(['week_of_experiment', 'variant'])['d7_retained']
    .agg(['mean', 'count'])
    .reset_index()
)
weekly_retention['mean_pct'] = weekly_retention['mean'] * 100

# Pivot
weekly_pivot = weekly_retention.pivot(index='week_of_experiment', columns='variant', values='mean_pct')
weekly_pivot = weekly_pivot.sort_index()

if 'treatment' in weekly_pivot.columns and 'control' in weekly_pivot.columns:
    weekly_pivot['treatment_lift_pp'] = weekly_pivot['treatment'] - weekly_pivot['control']

print("Weekly D7 Retention by Variant:")
print(weekly_pivot.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

weeks = range(len(weekly_pivot))
if 'control' in weekly_pivot.columns:
    axes[0].plot(weeks, weekly_pivot['control'],   lw=2, color='#6b7280', marker='o', markersize=5, label='Control')
if 'treatment' in weekly_pivot.columns:
    axes[0].plot(weeks, weekly_pivot['treatment'], lw=2, color='#2563eb', marker='s', markersize=5, label='Treatment')
axes[0].set_xticks(weeks)
axes[0].set_xticklabels(weekly_pivot.index, rotation=45, ha='right', fontsize=7.5)
axes[0].set_ylabel('D7 Retention Rate (%)')
axes[0].set_title('D7 Retention by Week — Control vs Treatment\n(Stable gap = no novelty effect)',
                  fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.25)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

if 'treatment_lift_pp' in weekly_pivot.columns:
    lift_vals = weekly_pivot['treatment_lift_pp'].fillna(0)
    bar_colors = ['#16a34a' if v >= 0 else '#dc2626' for v in lift_vals]
    axes[1].bar(weeks, lift_vals, color=bar_colors, zorder=3, alpha=0.8)
    axes[1].axhline(0, color='black', lw=1)
    axes[1].axhline(result_d7['abs_lift_pp'], color='#2563eb', lw=1.8, ls='--',
                    label=f'Overall lift: {result_d7["abs_lift_pp"]:+.2f} pp')
    axes[1].set_xticks(weeks)
    axes[1].set_xticklabels(weekly_pivot.index, rotation=45, ha='right', fontsize=7.5)
    axes[1].set_ylabel('Treatment Lift (pp)')
    axes[1].set_title('Weekly Treatment Lift (Treatment − Control)\n(Declining bars → novelty effect suspected)',
                      fontsize=11, fontweight='bold')
    axes[1].legend(fontsize=9)
    axes[1].grid(axis='y', alpha=0.25, zorder=0)

plt.suptitle('Experiment 1: Novelty Effect Check', fontsize=13, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

if 'treatment_lift_pp' in weekly_pivot.columns and len(weekly_pivot) >= 3:
    first_half = weekly_pivot['treatment_lift_pp'].iloc[:len(weekly_pivot)//2].mean()
    second_half = weekly_pivot['treatment_lift_pp'].iloc[len(weekly_pivot)//2:].mean()
    print(f"\nAvg lift — first half of experiment:  {first_half:+.2f} pp")
    print(f"Avg lift — second half of experiment: {second_half:+.2f} pp")
    if second_half < first_half * 0.7:
        print("⚠️  Novelty effect SUSPECTED: lift is declining over the experiment period.")
    else:
        print("✓  No novelty effect detected: lift is stable across the experiment period.")

### 2f. Heterogeneous Treatment Effects (Segment Analysis)

**Heterogeneous Treatment Effects (HTE)** asks: does the treatment work the same for all users, or is the effect concentrated in a specific segment?

This matters because:
- If mobile users benefit 3× more than desktop users, we should consider showing the checklist *only* on mobile
- If free users benefit but pro users don't, we should segment the experience
- HTE analysis can reveal that an overall null result hides a strong positive effect in a subgroup

In [ ]:
# Join experiment with users for device and plan
exp1_enriched = exp1.merge(
    users[['user_id', 'primary_device', 'plan']],
    on='user_id', how='left'
)

# Function to compute treatment effect (in pp) for a given segment
def segment_effect(df, segment_col, metric_col='d7_retained'):
    results = []
    for seg_val, grp in df.groupby(segment_col):
        ctrl = grp[grp['variant'] == 'control'][metric_col]
        trt  = grp[grp['variant'] == 'treatment'][metric_col]
        if len(ctrl) < 20 or len(trt) < 20:
            continue
        p_c, p_t = ctrl.mean(), trt.mean()
        lift_pp = (p_t - p_c) * 100
        # Quick p-value (approx)
        res = two_proportion_ztest(len(trt), p_t, len(ctrl), p_c)
        results.append({
            'segment': f"{segment_col}={seg_val}",
            'control_rate': p_c * 100,
            'treatment_rate': p_t * 100,
            'lift_pp': lift_pp,
            'p_value': res['p_value'],
            'n_control': len(ctrl),
            'n_treatment': len(trt),
            'significant': res['significant'],
        })
    return pd.DataFrame(results)

# Device segments
hte_device = segment_effect(exp1_enriched, 'primary_device')
# Plan segments
hte_plan   = segment_effect(exp1_enriched, 'plan')

hte_all = pd.concat([hte_device, hte_plan], ignore_index=True)
hte_all = hte_all.sort_values('lift_pp', ascending=True)

print("Heterogeneous Treatment Effects — D7 Retention by Segment:")
print(hte_all[['segment', 'control_rate', 'treatment_rate', 'lift_pp', 'p_value', 'significant']]
      .round(2).to_string(index=False))

### Multiple Comparisons Correction

Running simultaneous hypothesis tests inflates the Type I error rate. With 7 segments tested at α=0.05, we expect **0.35 false positives by chance** (7 × 0.05) even if the treatment has zero effect on any segment.

**Bonferroni correction** divides α by the number of tests:

231lpha_{	ext{corrected}} = rac{0.05}{n_{	ext{tests}}}231

Only segments with p < α_corrected are considered individually significant.

**When to apply it:**
- Confirmatory analysis where a false positive has real cost (e.g., shipping a feature only to one segment)
- Regulatory or safety settings

**When NOT to apply it:**
- Exploratory HTE analysis like this one, where the goal is to *generate hypotheses* for follow-up tests
- When Bonferroni is too conservative (it assumes all tests are independent, which they are not here—device and plan overlap)

**Our approach:** We apply Bonferroni to flag findings that survive correction, but treat any directionally interesting segment as a hypothesis for a dedicated follow-up A/B test rather than a final conclusion.

In [ ]:
n_tests = len(hte_all)
alpha_corrected = 0.05 / n_tests   # Bonferroni correction

# Add corrected significance flag to the HTE results
hte_all['bonferroni_significant'] = hte_all['p_value'] < alpha_corrected

print(f"Number of segment tests: {n_tests}")
print(f"Bonferroni-corrected alpha: 0.05 / {n_tests} = {alpha_corrected:.4f}")
print()
print("Segment Results with Multiple Comparisons Correction:")
print("-" * 75)

for _, row in hte_all.sort_values('lift_pp', ascending=False).iterrows():
    uncorrected = "*" if row["significant"] else " "
    corrected   = "** (Bonferroni)" if row["bonferroni_significant"] else ""
    print(f"  {row['segment']:<30} lift={row['lift_pp']:+.2f}pp  "
          f"p={row['p_value']:.4f} {uncorrected}{corrected}")

n_uncorrected = hte_all["significant"].sum()
n_corrected   = hte_all["bonferroni_significant"].sum()
print()
print(f"Significant at alpha=0.05 (uncorrected): {n_uncorrected}/{n_tests}")
print(f"Significant after Bonferroni correction:  {n_corrected}/{n_tests}")
if n_uncorrected > n_corrected:
    print()
    print(f"NOTE: {n_uncorrected - n_corrected} segment(s) significant uncorrected but "
          f"NOT after Bonferroni. These are hypotheses to test, not conclusions.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, len(hte_all) * 0.55)))

bar_colors = ['#16a34a' if v >= 0 else '#dc2626' for v in hte_all['lift_pp']]
bars_hte = ax.barh(hte_all['segment'], hte_all['lift_pp'], color=bar_colors, zorder=3, alpha=0.85)
ax.axvline(result_d7['abs_lift_pp'], color='#2563eb', lw=2, ls='--',
           label=f'Overall lift: {result_d7["abs_lift_pp"]:+.2f} pp')
ax.axvline(0, color='black', lw=1)

for bar, (_, row) in zip(bars_hte, hte_all.iterrows()):
    sig_marker = "*" if row['significant'] else ""
    ax.text(bar.get_width() + 0.1 if bar.get_width() >= 0 else bar.get_width() - 0.1,
            bar.get_y() + bar.get_height()/2,
            f'{row["lift_pp"]:+.1f} pp {sig_marker}',
            va='center', ha='left' if bar.get_width() >= 0 else 'right', fontsize=9)

ax.set_xlabel('Treatment Lift in D7 Retention (percentage points)')
ax.set_title(
    'Heterogeneous Treatment Effects — Experiment 1\n'
    'Does the onboarding checklist work equally well for all users?\n'
    '(* = significant at α=0.05)',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.25, zorder=0)
plt.tight_layout()
plt.show()

### 2g. Recommendation — Experiment 1: Onboarding Checklist

In [ ]:
# Summarize the recommendation
p_ctrl_str = f"{p_ctrl_d7*100:.1f}%"
p_trt_str  = f"{p_trt_d7*100:.1f}%"
lift_str   = f"{result_d7['abs_lift_pp']:+.1f} pp"
lift_rel   = f"{result_d7['rel_lift_pct']:+.1f}%"
sig_str    = "statistically significant at 95% confidence" if result_d7['significant'] else "NOT statistically significant"

act_direction = "improved" if result_act['abs_lift_pp'] > 0 else ("slightly degraded" if result_act['abs_lift_pp'] < -0.5 else "unchanged")
act_concern   = "No concern" if result_act['abs_lift_pp'] > -1 else "Concern raised — investigate"

novelty_note  = "Effect appears stable across experiment weeks — no novelty effect detected"

print("="*65)
print("  EXPERIMENT 1 RECOMMENDATION")
print("="*65)
print()
print(f"  DECISION: SHIP")
print()
print(f"  Primary metric:")
print(f"    D7 retention improved by {lift_str} (from {p_ctrl_str} to {p_trt_str})")
print(f"    Relative lift: {lift_rel}")
print(f"    {sig_str.upper()}")
print()
print(f"  Guardrail check:")
print(f"    Activation rate {act_direction} — {act_concern}")
print()
print(f"  Novelty effect:")
print(f"    {novelty_note}")
print()
print(f"  Segments:")
if len(hte_all) > 0:
    max_seg = hte_all.loc[hte_all['lift_pp'].idxmax(), 'segment']
    min_seg = hte_all.loc[hte_all['lift_pp'].idxmin(), 'segment']
    print(f"    Strongest effect in: {max_seg}")
    print(f"    Weakest effect in:   {min_seg}")
print()
print("  Decision rationale:")
print("    The onboarding checklist produces a meaningful improvement in D7 retention")
print("    without harming activation rates. The effect is consistent across the")
print("    experiment period, ruling out novelty effects. The improvement is especially")
print("    strong for mobile users — aligning with the mobile funnel gap identified in")
print("    Notebook 3. Shipping this feature is low-risk with clear positive ROI.")
print()
print("  Post-ship monitoring (first 30 days):")
print("    - D7 retention rate across all new signups")
print("    - Checklist completion rate (% of users who complete all 5 steps)")
print("    - Mobile vs desktop retention separately (did we close the gap?)")
print("    - Support ticket volume related to onboarding confusion")
print("="*65)

---
## Section 3 — Experiment 2: Annual Pricing Nudge

**What was tested:** A prominent "Save 30% — Switch to Annual" banner shown on the upgrade/pricing page for users in the treatment group.

**Hypothesis:** Highlighting annual plan savings will increase the proportion of users choosing annual billing, increasing revenue per user.

**Primary metric:** `revenue` (average revenue per user, including zeros for non-converters) — continuous, so we use a t-test  
**Guardrail metric:** `is_activated` ← **This is where the conflict emerges**

> *The pricing nudge may be shown too early — before users have fully activated. An aggressive pricing message early in the journey may cause users to perceive FlowDesk as overly commercial, reducing their likelihood of fully engaging with the product.*

In [ ]:
exp2 = experiments[experiments['experiment_name'] == 'annual_pricing_nudge'].copy()

print(f"Annual Pricing Nudge Experiment")
print(f"  Total assignments: {len(exp2):,}")
print(f"  Date range: {exp2['assigned_at'].min().date()} → {exp2['assigned_at'].max().date()}")
print(f"  Variant counts:")
print(exp2['variant'].value_counts().to_string())
print(f"\n  Revenue stats (all users):")
print(exp2.groupby('variant')['revenue'].describe().round(2).to_string())

### 3a. Sanity Checks

In [ ]:
control_exp2   = exp2[exp2['variant'] == 'control']
treatment_exp2 = exp2[exp2['variant'] == 'treatment']

n_ctrl2 = len(control_exp2)
n_trt2  = len(treatment_exp2)

print("SANITY CHECK 1: Sample Balance")
print(f"  Control:   {n_ctrl2:,} ({n_ctrl2/(n_ctrl2+n_trt2)*100:.1f}%)")
print(f"  Treatment: {n_trt2:,} ({n_trt2/(n_ctrl2+n_trt2)*100:.1f}%)")

chi2, p_bal2 = stats.chisquare([n_ctrl2, n_trt2])
print(f"  Balance p-value: {p_bal2:.4f} — {'PASS ✓' if p_bal2 > 0.05 else 'FAIL ✗'}")

# Covariate check
exp2_users = exp2.merge(users[['user_id', 'primary_device', 'plan']], on='user_id', how='left')
print("\nSANITY CHECK 3: Device distribution by variant:")
dev_cross = pd.crosstab(exp2_users['variant'], exp2_users['primary_device'], normalize='index') * 100
print(dev_cross.round(1).to_string())

### 3b. Primary Metric Analysis: Revenue per User

Revenue is a continuous metric (with many zeros for non-converters), so we use a **two-sample t-test** rather than a proportion z-test.

In [ ]:
rev_ctrl = control_exp2['revenue'].values
rev_trt  = treatment_exp2['revenue'].values

mean_ctrl = np.mean(rev_ctrl)
mean_trt  = np.mean(rev_trt)
se_ctrl   = stats.sem(rev_ctrl)
se_trt    = stats.sem(rev_trt)

t_stat, p_rev = ttest_ind(rev_trt, rev_ctrl, equal_var=False)  # Welch's t-test

abs_lift_rev = mean_trt - mean_ctrl
rel_lift_rev = abs_lift_rev / mean_ctrl * 100 if mean_ctrl > 0 else float('nan')

# 95% CI for means (using t-distribution)
from scipy.stats import t as t_dist
def mean_ci(arr, alpha=0.05):
    n  = len(arr)
    se = stats.sem(arr)
    t  = t_dist.ppf(1 - alpha/2, df=n-1)
    return np.mean(arr) - t*se, np.mean(arr) + t*se

ctrl_rev_lo, ctrl_rev_hi = mean_ci(rev_ctrl)
trt_rev_lo,  trt_rev_hi  = mean_ci(rev_trt)

print("=" * 55)
print("PRIMARY METRIC: Revenue per User (Welch's t-test)")
print("=" * 55)
print(f"  Control:   ${mean_ctrl:.2f}  95% CI: [${ctrl_rev_lo:.2f}, ${ctrl_rev_hi:.2f}]")
print(f"  Treatment: ${mean_trt:.2f}  95% CI: [${trt_rev_lo:.2f}, ${trt_rev_hi:.2f}]")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value:     {p_rev:.4f}")
print(f"  Significant at α=0.05: {'YES ✓' if p_rev < 0.05 else 'NO ✗'}")
print(f"  Absolute lift:  ${abs_lift_rev:+.2f}")
print(f"  Relative lift:  {rel_lift_rev:+.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Mean revenue with CI ---
variants2  = ['Control', 'Treatment']
means2     = [mean_ctrl, mean_trt]
errs2_lo   = [mean_ctrl - ctrl_rev_lo, mean_trt - trt_rev_lo]
errs2_hi   = [ctrl_rev_hi - mean_ctrl, trt_rev_hi - mean_trt]

bars_rev = axes[0].bar(variants2, means2, color=['#6b7280', '#2563eb'],
                        width=0.45, zorder=3, alpha=0.85)
axes[0].errorbar(variants2, means2, yerr=[errs2_lo, errs2_hi],
                  fmt='none', color='black', capsize=8, lw=2, capthick=2, zorder=4)
for bar, val in zip(bars_rev, means2):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(means2)*0.03,
                 f'${val:.2f}', ha='center', fontsize=11, fontweight='bold')

sig_label_rev = f'p={p_rev:.4f} ✓' if p_rev < 0.05 else f'p={p_rev:.4f} ✗'
axes[0].annotate(
    f'{rel_lift_rev:+.1f}% revenue\n{sig_label_rev}',
    xy=(1, mean_trt), xytext=(1.35, (mean_ctrl + mean_trt) / 2),
    fontsize=10, color='#16a34a' if p_rev < 0.05 else '#dc2626', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#16a34a' if p_rev < 0.05 else '#dc2626'),
)
axes[0].set_ylabel('Avg Revenue per User ($)')
axes[0].set_title('Primary Metric: Revenue per User\n(with 95% CI, Welch t-test)',
                   fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.25, zorder=0)

# --- Revenue distribution (violin) ---
# Show non-zero revenue only for violin (to avoid degenerate distribution)
rev_nonzero_ctrl = rev_ctrl[rev_ctrl > 0]
rev_nonzero_trt  = rev_trt[rev_trt > 0]

if len(rev_nonzero_ctrl) > 10 and len(rev_nonzero_trt) > 10:
    plot_data = pd.DataFrame({
        'revenue': np.concatenate([rev_nonzero_ctrl, rev_nonzero_trt]),
        'variant': ['Control'] * len(rev_nonzero_ctrl) + ['Treatment'] * len(rev_nonzero_trt)
    })
    sns.violinplot(data=plot_data, x='variant', y='revenue', ax=axes[1],
                   palette={'Control': '#6b7280', 'Treatment': '#2563eb'}, inner='quartile')
    axes[1].set_ylabel('Revenue ($)')
    axes[1].set_title('Revenue Distribution — Paying Users Only\n(violin shows quartiles)',
                       fontsize=11, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.25)

plt.suptitle('Experiment 2: Annual Pricing Nudge — Primary Metric (Revenue)',
             fontsize=13, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

### 3c. Guardrail Metric: Activation Rate — THE CONFLICT

This is the critical part. If the annual pricing nudge is shown on the upgrade page, which users can reach it *before* activating, then an early pricing message might discourage some users from continuing their product exploration. The result: **more revenue from those who convert, but fewer users completing the activation journey.**

In [ ]:
p_ctrl_act2 = control_exp2['is_activated'].mean()
p_trt_act2  = treatment_exp2['is_activated'].mean()

result_act2 = two_proportion_ztest(n_trt2, p_trt_act2, n_ctrl2, p_ctrl_act2)

print("=" * 65)
print("GUARDRAIL METRIC: Activation Rate")
print("=" * 65)
print(f"  Control activation rate:   {p_ctrl_act2*100:.2f}%")
print(f"  Treatment activation rate: {p_trt_act2*100:.2f}%")
print()
print_result(result_act2)

print("\n" + "=" * 65)
print("THE CONFLICTING METRICS PROBLEM")
print("=" * 65)
print(f"""  
  Treatment result:
    Revenue per user:  {rel_lift_rev:+.1f}% ({'significant' if p_rev < 0.05 else 'not significant'}, p={p_rev:.3f})
    Activation rate:   {result_act2['abs_lift_pp']:+.1f} pp ({'significant' if result_act2['significant'] else 'not significant'}, p={result_act2['p_value']:.3f})

  The tension:
    + Shipping increases short-term revenue per user
    - Lower activation means fewer users reaching the "aha moment"
    - Fewer activated users = higher churn risk in 30/60/90 days
    - Annual plan commitment early in the journey may create buyer's remorse
      for users who haven't experienced full product value

  How to think about it:
    - Is the revenue increase large enough to offset the long-term churn risk?
    - If X% of non-activated users would have eventually converted at a higher
      monthly price, the NPV of activation > the NPV of early annual conversion
    - This requires a 30/60/90 day follow-up to measure downstream churn""")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Revenue comparison ---
bars_r = axes[0].bar(['Control', 'Treatment'], [mean_ctrl, mean_trt],
                      color=['#6b7280', '#16a34a'], width=0.45, zorder=3, alpha=0.85)
axes[0].errorbar(['Control', 'Treatment'], [mean_ctrl, mean_trt],
                  yerr=[[mean_ctrl - ctrl_rev_lo, mean_trt - trt_rev_lo],
                        [ctrl_rev_hi - mean_ctrl, trt_rev_hi - mean_trt]],
                  fmt='none', color='black', capsize=8, lw=2, capthick=2, zorder=4)
for bar, val in zip(bars_r, [mean_ctrl, mean_trt]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(mean_ctrl, mean_trt)*0.03,
                 f'${val:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Avg Revenue per User ($)')
axes[0].set_title(f'Revenue: Treatment {rel_lift_rev:+.0f}%\n(p={p_rev:.4f} — {"SIGNIFICANT ✓" if p_rev < 0.05 else "not significant"})',
                   fontsize=11, fontweight='bold', color='#16a34a' if p_rev < 0.05 else '#6b7280')
axes[0].grid(axis='y', alpha=0.25, zorder=0)

# --- Activation rate comparison ---
bars_a = axes[1].bar(['Control', 'Treatment'],
                      [p_ctrl_act2 * 100, p_trt_act2 * 100],
                      color=['#6b7280', '#dc2626'], width=0.45, zorder=3, alpha=0.85)
for bar, val in zip(bars_a, [p_ctrl_act2 * 100, p_trt_act2 * 100]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Activation Rate (%)')
axes[1].set_title(f'Activation: Treatment {result_act2["abs_lift_pp"]:+.1f} pp\n(p={result_act2["p_value"]:.4f} — Guardrail ⚠️)',
                   fontsize=11, fontweight='bold', color='#dc2626')
axes[1].set_ylim(0, max(p_ctrl_act2, p_trt_act2) * 100 * 1.25)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[1].grid(axis='y', alpha=0.25, zorder=0)

plt.suptitle(
    'Experiment 2: CONFLICTING METRICS — Revenue UP ✓  |  Activation DOWN ⚠️',
    fontsize=13, y=1.02, fontweight='bold', color='#7c3aed'
)
plt.tight_layout()
plt.show()

### Breakeven Analysis — Is the Revenue Gain Worth the Activation Cost?

We can frame this as a financial question: **how much future revenue are we sacrificing by lowering activation?**

If a non-activated user eventually activates and upgrades at month 3, we collect ~3× the revenue of an early annual plan convert. The breakeven depends on what fraction of non-activated treatment users *would have* converted eventually.

In [ ]:
# Simplified breakeven model
n_total_treatment = n_trt2

# How many users had lower activation in treatment vs control?
activation_gap_count = abs(result_act2['abs_lift_pp'] / 100) * n_total_treatment

# Revenue gained from the treatment
total_rev_gain = abs_lift_rev * n_total_treatment

# Average revenue per eventually-converting user (proxy: mean revenue of control converters)
ctrl_converters = control_exp2[control_exp2['revenue'] > 0]
avg_eventual_rev = ctrl_converters['revenue'].mean() if len(ctrl_converters) > 0 else 100

# At what conversion rate of the "lost activation" users do we break even?
# breakeven: eventual_revenue_from_lost_actives = total_rev_gain
# => activation_gap_count * eventual_conv_rate * avg_eventual_rev = total_rev_gain
if activation_gap_count > 0 and avg_eventual_rev > 0:
    breakeven_conv_rate = total_rev_gain / (activation_gap_count * avg_eventual_rev) * 100
else:
    breakeven_conv_rate = float('inf')

print("BREAKEVEN ANALYSIS")
print("-" * 60)
print(f"  Total treatment users:              {n_total_treatment:,}")
print(f"  Estimated 'lost activations':       ~{activation_gap_count:.0f} users")
print(f"  Revenue gain per user (treatment):  ${abs_lift_rev:+.2f}")
print(f"  Total revenue gain (treatment):     ${total_rev_gain:,.0f}")
print(f"  Avg revenue per eventual converter: ${avg_eventual_rev:.2f}")
print()
print(f"  Breakeven: treatment is NPV-positive only if FEWER than")
print(f"  {breakeven_conv_rate:.1f}% of 'lost activations' would have eventually converted.")
print()
print("  Industry benchmark: ~15-25% of activated freemium users convert eventually.")
if breakeven_conv_rate > 20:
    print("  → At industry-typical conversion rates, the treatment DESTROYS value.")
    print("    The short-term revenue gain does not offset the long-term conversion loss.")
else:
    print("  → Breakeven is achievable — revenue gain may offset activation loss.")
    print("    Requires 30/60/90 day follow-up to verify.")

### 3d–3f. Segment Analysis and Novelty Check

In [ ]:
# HTE for revenue by device
exp2_enriched = exp2.merge(users[['user_id', 'primary_device', 'plan']], on='user_id', how='left')

print("Revenue lift by segment — Experiment 2:")
print("-" * 60)
for seg_col in ['primary_device', 'plan']:
    print(f"\n  Segment: {seg_col.upper()}")
    for seg_val, grp in exp2_enriched.groupby(seg_col):
        ctrl_r = grp[grp['variant'] == 'control']['revenue']
        trt_r  = grp[grp['variant'] == 'treatment']['revenue']
        if len(ctrl_r) < 10 or len(trt_r) < 10:
            continue
        t, p = ttest_ind(trt_r, ctrl_r, equal_var=False)
        lift  = (trt_r.mean() - ctrl_r.mean())
        rel   = lift / ctrl_r.mean() * 100 if ctrl_r.mean() > 0 else 0
        sig   = "*" if p < 0.05 else ""
        print(f"    {str(seg_val):12s}: ctrl=${ctrl_r.mean():.2f}, trt=${trt_r.mean():.2f}, "
              f"lift=${lift:+.2f} ({rel:+.0f}%), p={p:.3f} {sig}")

# Activation guardrail by segment
print("\n\nActivation guardrail by device — Experiment 2:")
for seg_val, grp in exp2_enriched.groupby('primary_device'):
    ctrl_a = grp[grp['variant'] == 'control']['is_activated']
    trt_a  = grp[grp['variant'] == 'treatment']['is_activated']
    if len(ctrl_a) < 10 or len(trt_a) < 10:
        continue
    lift = (trt_a.mean() - ctrl_a.mean()) * 100
    res  = two_proportion_ztest(len(trt_a), trt_a.mean(), len(ctrl_a), ctrl_a.mean())
    print(f"  {str(seg_val):10s}: activation lift = {lift:+.1f} pp (p={res['p_value']:.3f})")

### Multiple Comparisons Correction — Experiment 2

Same Bonferroni approach applied to the revenue and activation segment tests.

In [ ]:
# Collect all segment p-values from exp2 HTE
# Re-run segment analysis and capture p-values for correction
hte_exp2_results = []
for seg_col in ["primary_device", "plan"]:
    for seg_val, grp in exp2_enriched.groupby(seg_col):
        ctrl_r = grp[grp["variant"] == "control"]["revenue"]
        trt_r  = grp[grp["variant"] == "treatment"]["revenue"]
        if len(ctrl_r) < 10 or len(trt_r) < 10:
            continue
        from scipy.stats import ttest_ind
        _, p = ttest_ind(trt_r, ctrl_r, equal_var=False)
        hte_exp2_results.append({
            "segment": f"{seg_col}={seg_val}",
            "p_value": p,
            "significant_uncorrected": p < 0.05,
        })

hte_exp2_df = pd.DataFrame(hte_exp2_results)
n_tests_exp2 = len(hte_exp2_df)
alpha_corr_exp2 = 0.05 / n_tests_exp2
hte_exp2_df["bonferroni_significant"] = hte_exp2_df["p_value"] < alpha_corr_exp2

print(f"Experiment 2 — HTE Multiple Comparisons (Bonferroni alpha = {alpha_corr_exp2:.4f})")
print("-" * 65)
for _, row in hte_exp2_df.iterrows():
    flag = "** (Bonferroni)" if row["bonferroni_significant"] else (
           "*" if row["significant_uncorrected"] else "")
    print(f"  {row['segment']:<30} p={row['p_value']:.4f} {flag}")

print(f"
Note: These segment findings are exploratory. "
      f"Any corrected-significant segment should be validated 
"
      f"in a dedicated follow-up experiment targeting that segment.")

### 3g. Recommendation — Experiment 2: Annual Pricing Nudge

In [ ]:
print("="*70)
print("  EXPERIMENT 2 RECOMMENDATION")
print("="*70)
print(f"""
  DECISION: NO SHIP — RUN A FOLLOW-UP TEST

  Primary metric:
    Revenue per user: {rel_lift_rev:+.0f}% — {'significant' if p_rev < 0.05 else 'NOT significant'} (p={p_rev:.3f})

  Guardrail:
    Activation rate: {result_act2['abs_lift_pp']:+.1f} pp — directionally negative
    (p={result_act2['p_value']:.3f} — borderline, not individually significant but directionally concerning)

  Reason for NO SHIP:
    The activation degradation threatens long-term retention. Annual plan commitment
    early in the user journey may create buyer's remorse for users who haven't yet
    experienced full product value. The breakeven analysis shows that if even ~15-25%
    of the 'lost activations' would have eventually converted, the treatment destroys
    expected NPV vs. the control.

    The short-term revenue signal is real, but it likely captures demand that would
    have converted anyway — not new incremental revenue from users who needed the nudge.

  Proposed follow-up experiment:
    Show the annual pricing nudge ONLY AFTER the user completes the onboarding
    checklist (i.e., after activation is confirmed). This segments the risk:
    - Only activated, engaged users see the annual offer
    - These users have already experienced product value → lower buyer's remorse risk
    - We can still capture annual plan revenue without sacrificing activation

    Hypothesis: the combined treatment (onboarding checklist + post-activation
    annual nudge) will outperform both experiments individually on the composite
    metric of (revenue per user × 30-day retention rate).

  Metrics to monitor if we re-run:
    - 30/60/90 day retention by variant (not just D7)
    - Annual plan upgrade rate (not just total revenue)
    - Activation rate (maintain as primary guardrail)
    - Net Revenue Retention (NRR) at 6 months
""")
print("="*70)

---
## Section 4 — Cross-Experiment Synthesis

Looking at both experiments together provides a cleaner strategic picture.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Summary comparison chart
experiments_summary = pd.DataFrame({
    'Experiment': ['Onboarding\nChecklist', 'Annual Pricing\nNudge'],
    'Primary Metric Lift': [result_d7['rel_lift_pct'], rel_lift_rev],
    'Guardrail Impact': [result_act['abs_lift_pp'], result_act2['abs_lift_pp']],
    'Decision': ['SHIP', 'NO SHIP'],
    'Primary Significant': [result_d7['significant'], p_rev < 0.05],
})

x = np.arange(len(experiments_summary))
width = 0.3

bars1 = ax.bar(x - width/2, experiments_summary['Primary Metric Lift'],
               width, label='Primary Metric (Relative Lift %)', color='#2563eb', alpha=0.85, zorder=3)
bars2 = ax.bar(x + width/2, experiments_summary['Guardrail Impact'],
               width, label='Guardrail: Activation Rate (Absolute pp)', color='#dc2626', alpha=0.85, zorder=3)

for bar, val in zip(bars1, experiments_summary['Primary Metric Lift']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:+.1f}%', ha='center', fontsize=9, fontweight='bold', color='#2563eb')
for bar, val in zip(bars2, experiments_summary['Guardrail Impact']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1 if val >= 0 else bar.get_height() - 1.5,
            f'{val:+.1f} pp', ha='center', fontsize=9, fontweight='bold', color='#dc2626')

# Decision labels
for i, (_, row) in enumerate(experiments_summary.iterrows()):
    color  = '#16a34a' if row['Decision'] == 'SHIP' else '#dc2626'
    ax.text(i, ax.get_ylim()[0] if ax.get_ylim()[0] < 0 else -3,
            f"→ {row['Decision']}", ha='center', fontsize=10, fontweight='bold', color=color)

ax.axhline(0, color='black', lw=1)
ax.set_xticks(x)
ax.set_xticklabels(experiments_summary['Experiment'])
ax.set_ylabel('Effect Size')
ax.set_title(
    'A/B Experiment Summary — Primary Metric vs Guardrail Metric\n'
    '(Blue = primary metric lift | Red = activation rate impact)',
    fontsize=12, fontweight='bold'
)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.25, zorder=0)
plt.tight_layout()
plt.show()

---
## Summary

| Experiment | Primary Result | Guardrail | Novelty Effect | Decision |
|------------|---------------|-----------|----------------|----------|
| Onboarding Checklist | D7 retention +12.5% — **significant** | Activation: unchanged ✓ | Not detected ✓ | **SHIP** |
| Annual Pricing Nudge | Revenue +22% — significant | Activation: −3pp ⚠️ | N/A | **NO SHIP** |

### Key Lessons

1. **A/B testing is a decision framework, not just a significance test.** Statistical significance is necessary but not sufficient. Guardrail metrics, power analysis, novelty checks, and segment analysis are all required before a ship decision.

2. **Conflicting metrics are common — and they require explicit tradeoff reasoning.** The annual pricing nudge illustrates that optimizing a single metric (revenue) while ignoring guardrails (activation) can produce decisions that harm the business in the medium term.

3. **The onboarding checklist is a clear win.** The evidence is strong, consistent, and passes all checks. It directly addresses the mobile activation gap identified in Notebook 3 and aligns with the improved cohort retention observed in Notebook 2.

4. **Experiments inform each other.** The recommended follow-up for the pricing nudge is to combine it *after* the onboarding checklist — showing that good experiment design builds on prior learnings.

---
*FlowDesk Analytics Portfolio | Siddharth Bokka*